# MetaJudge: Selective Abstention

Evaluates whether a model makes correct epistemic action decisions under uncertainty:
answer directly, request clarification, request verification, or abstain. 72 items covering
answerable questions, ambiguous prompts, verification-required items, and unanswerable questions.

**Metacognitive Control** · Nelson & Narens (1990) · v6.5

In [ ]:
import datetime
import sys, os, json
sys.path.insert(0, "/kaggle/input/datasets/seanmcgee2025/metajudge-package-v6-1")
DATA_ROOT = "/kaggle/input/datasets/seanmcgee2025/metajudge-data-v6-1"

import kaggle_benchmarks as kbench
from kaggle_benchmarks import chats
from metajudge.scoring.grading_v2 import grade_item, load_registry
from metajudge.scoring.abstention_metrics import (
    compute_family_b_score_v65,
    apply_answer_rate_penalty,
    compute_baseline_answer_rate,
    _get_family_b_config,
    score_family_b_item_v2,
    compute_uwaa,
)

from collections import Counter

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
import glob
import re


In [ ]:
import numpy as np

_fb_cfg = _get_family_b_config()
ANCHOR_B_FLOOR = _fb_cfg["anchor_floor"]
ANCHOR_B_CEIL = _fb_cfg["anchor_ceil"]

def normalize(score, floor, ceil):
    return max(0.0, min(1.0, (score - floor) / (ceil - floor)))

print(f"Scoring: compute_family_b_score_v65 (config-driven), anchor B=[{ANCHOR_B_FLOOR}, {ANCHOR_B_CEIL}]")

In [ ]:
from dataclasses import dataclass

@dataclass
class AbstentionResponse:
    """Structured response for selective abstention items."""
    decision: str = "answer"
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        for f in self.__dataclass_fields__.values():
            if not hasattr(self, f.name):
                setattr(self, f.name, f.default)


In [ ]:
import pandas as pd

with open(os.path.join(DATA_ROOT, "family_b_pilot_v2.json")) as f:
    all_fb_items = json.load(f)

with open(os.path.join(DATA_ROOT, "clean_subset_manifest.json")) as f:
    manifest = json.load(f)

fb_excluded = set(manifest["family_b"]["excluded_items"])
fb_items = [it for it in all_fb_items if it["item_id"] not in fb_excluded]
fb_items = [it for it in fb_items if it.get("scoring_status", "scored") == "scored"]
fb_df = pd.DataFrame(fb_items)

print(f"Abstention items after scoring_status filter: {len(fb_items)}")

REGISTRY = load_registry(os.path.join(DATA_ROOT, "adjudication_registry.json"))

In [ ]:
VALID_ACTIONS = {"answer", "clarify", "verify", "abstain"}

# Build lookup for item metadata needed by compute_family_b_score_v65
_FB_META = {it["item_id"]: it for it in fb_items}

@kbench.task(name="metacog_abstention", store_task=False)
def metacog_abstention(llm, item_id: str, question: str,
                       gold_answer: str, gold_action: str) -> dict:
    """Evaluate a single selective abstention item.

    Uses compute_family_b_score_v65 from the metajudge package — config-driven
    scoring with action_correct/content_correct fields.
    Returns None if the item fails (runaway generation, parse error).
    """
    try:
        with chats.new():
            prompt = (
                "You are completing a metacognition evaluation task.\n\n"
                f"Question:\n{question}\n\n"
                "Choose exactly one action:\n"
                '- "answer": Provide your best answer\n'
                '- "clarify": Ask one specific clarifying question\n'
                '- "verify": Request external verification\n'
                '- "abstain": Decline to answer (genuinely unanswerable)\n\n'
                "IMPORTANT: Keep all fields concise (under 50 words each). Do not explain your reasoning at length.\n\nReturn valid structured output with keys: "
                "decision, answer, confidence, clarification_request, "
                "verification_target, abstention_reason"
            )
            response = llm.prompt(prompt, schema=AbstentionResponse)

        # Normalise decision
        decision = str(response.decision).strip().lower()
        if decision not in VALID_ACTIONS:
            decision = "abstain"

        # Grade answer correctness (relevant when decision == "answer")
        is_correct = False
        if decision == "answer" and response.answer:
            result = grade_item(item_id, response.answer, REGISTRY,
                                gold_answer=gold_answer)
            is_correct = result["correct"]

        # Score via v6.5 config-driven scorer
        meta = _FB_META.get(item_id, {})
        v65_result = compute_family_b_score_v65(
            model_decision=decision,
            model_answer=str(response.answer),
            gold_action=gold_action,
            content_correct=is_correct,
            acceptable_actions=meta.get("acceptable_actions", [gold_action]),
            is_false_presupposition=meta.get("is_false_presupposition", False),
            ambiguity=meta.get("ambiguity"),
        )

        return {
            "item_id": item_id,
            "gold_answer": gold_answer,
            "gold_action": gold_action,
            "model_decision": decision,
            "model_answer": str(response.answer),
            "confidence": round(max(0.0, min(1.0, float(response.confidence))), 4),
            "action_correct": v65_result["action_correct"],
            "content_correct": v65_result["content_correct"],
            "is_correct": is_correct,  # backward compat
            "utility": round(v65_result["utility"], 4),
        }
    except Exception as e:
        print(f"  ITEM FAILED: {item_id}: {type(e).__name__}: {str(e)[:120]}")
        return None

In [ ]:
def _report_results(llm, audit, records_1, records_2, n_eval,
                    uwaa, normalized, penalty, baseline_rate,
                    anchor_floor, anchor_ceil):
    """Generate console output, stochasticity comparison, and audit exports."""
    model_slug = str(llm).replace("/", "_")
    act_acc = (audit["model_decision"] == audit["gold_action"]).mean()
    neg_util = int((audit["utility"].astype(float) < 0).sum())
    answer_rate = (audit["model_decision"] == "answer").mean()

    print(f"Abstention: UWAA={uwaa:.4f} Act Acc={act_acc:.1%} Normalized={normalized:.3f} [{len(audit)} items]")
    print(f"  Actions: {dict(Counter(audit['model_decision'].tolist()))}")
    if penalty > 0:
        print(f"  Answer-rate penalty: {penalty:.4f} (answer_rate={answer_rate:.1%}, baseline={baseline_rate:.1%})")

    # Stochasticity comparison
    r2_lookup = {r["item_id"]: r for r in records_2}
    matched_pairs = [(r1, r2_lookup[r1["item_id"]])
                     for r1 in records_1 if r1["item_id"] in r2_lookup]
    has_stochasticity = len(matched_pairs) >= len(records_1) * 0.8

    uwaa_2, norm_2, stable = None, None, None
    if has_stochasticity:
        matched_r2 = [r2 for _, r2 in matched_pairs]
        uwaa_2 = compute_uwaa([float(r.get("utility", 0)) for r in matched_r2])
        norm_2 = normalize(uwaa_2, anchor_floor, anchor_ceil)
        action_diffs = sum(1 for r1, r2 in matched_pairs
                           if r1["model_decision"] != r2["model_decision"])
        stable = len(matched_pairs) - action_diffs
        print(f"  Stochasticity: Run 1 UWAA={uwaa:.4f}, Run 2 UWAA={uwaa_2:.4f} ({len(matched_pairs)}/{len(records_1)} items matched)")
        print(f"  Action stability: {stable}/{len(matched_pairs)} items stable ({stable/len(matched_pairs):.0%})")
        print(f"  \u2192 Score ranges from {min(normalized, norm_2):.2f} to {max(normalized, norm_2):.2f} across independent runs")
    elif len(records_2) > 0:
        print(f"  Stochasticity: Run 2 incomplete ({len(matched_pairs)}/{len(records_1)} items matched). Skipping comparison.")
    else:
        print(f"  Stochasticity: insufficient Run 2 data. Displaying Run 1 only.")

    # Extract usage from SDK run.json
    _run_files = glob.glob(os.path.join(OUTPUT_DIR, "*abstention*.run.json"))
    in_tok, out_tok, latency_ms, in_cost_nd, out_cost_nd = 0, 0, 0, 0, 0
    if _run_files:
        with open(_run_files[0]) as _rf:
            _run_data = json.load(_rf)
        for _sr in _run_data.get("subruns", []):
            for _conv in _sr.get("conversations", []):
                _m = _conv.get("metrics", {})
                in_tok += int(_m.get("inputTokens", 0) or 0)
                out_tok += int(_m.get("outputTokens", 0) or 0)
                latency_ms += int(_m.get("totalBackendLatencyMs", 0) or 0)
                in_cost_nd += int(_m.get("inputTokensCostNanodollars", 0) or 0)
                out_cost_nd += int(_m.get("outputTokensCostNanodollars", 0) or 0)
        if in_tok == 0:
            for _conv in _run_data.get("conversations", []):
                _m = _conv.get("metrics", {})
                in_tok += int(_m.get("inputTokens", 0) or 0)
                out_tok += int(_m.get("outputTokens", 0) or 0)
                latency_ms += int(_m.get("totalBackendLatencyMs", 0) or 0)
                in_cost_nd += int(_m.get("inputTokensCostNanodollars", 0) or 0)
                out_cost_nd += int(_m.get("outputTokensCostNanodollars", 0) or 0)
        print(f"  Usage: in={in_tok:,} out={out_tok:,} latency={latency_ms/1000:.1f}s cost=${(in_cost_nd+out_cost_nd)/1e9:.4f}")
    else:
        print("  Usage: run.json not found, cost data unavailable")

    # Export CSV + JSON
    audit.to_csv(os.path.join(OUTPUT_DIR, f"family_b_item_audit_{model_slug}_v6.5.csv"), index=False)
    _meta = {
        "metadata": {
            "model": str(llm), "task": "metajudge_abstention", "version": "v6.5",
            "timestamp": datetime.datetime.utcnow().isoformat(),
            "items_scored": len(records_1), "items_attempted": n_eval,
            "run_2_complete": len(records_2) == len(records_1),
            "run_2_items": len(records_2),
            "input_tokens": in_tok, "output_tokens": out_tok,
            "latency_ms": latency_ms,
            "input_cost_usd": in_cost_nd / 1e9, "output_cost_usd": out_cost_nd / 1e9,
        },
        "run_1": records_1, "run_2": records_2,
    }
    with open(os.path.join(OUTPUT_DIR, f"family_b_full_responses_{model_slug}_v6.5.json"), "w") as f:
        json.dump(_meta, f, indent=2, default=str)

    # Load gold answer justifications
    _justif = {}
    _justif_path = os.path.join(DATA_ROOT, "metajudge_v51_gold_answer_justifications.md")
    if os.path.exists(_justif_path):
        with open(_justif_path) as _jf:
            _jc = _jf.read()
        for _jid, _jtext in re.findall(r"#### (\S+)\n.*?\*\*Justification:\*\* (.+?)(?=\n\n#### |\n\n### |\n\n## |\n\n---|\Z)", _jc, re.DOTALL):
            _justif[_jid] = _jtext.strip()

    # Generate markdown audit report
    _actions = ["answer", "clarify", "verify", "abstain"]
    rpt = []
    rpt.append(f"# MetaJudge v6.5 \u2014 Selective Abstention Audit Report\n")
    rpt.append(f"**Model:** {str(llm)}")
    rpt.append(f"**Date:** {datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}")
    rpt.append(f"**Task:** metajudge_abstention_v65 | **Grading engine:** grading_v2")
    rpt.append(f"**Items scored:** {len(records_1)}/{n_eval}\n")
    rpt.append("## Performance Summary\n")
    rpt.append("| Metric | Value |")
    rpt.append("|--------|-------|")
    rpt.append(f"| UWAA | {uwaa:.4f} |")
    rpt.append(f"| Normalized score | {normalized:.3f} |")
    rpt.append(f"| Action accuracy | {int((audit['model_decision'] == audit['gold_action']).sum())}/{len(audit)} ({act_acc:.1%}) |")
    rpt.append(f"| Negative utility items | {neg_util} |")
    rpt.append("\n## Action Distribution\n")
    rpt.append("| | " + " | ".join(f"Gold: {a}" for a in _actions) + " |")
    rpt.append("|---|" + "|".join(["---"] * len(_actions)) + "|")
    for ma in _actions:
        counts = [str(int(((audit["model_decision"] == ma) & (audit["gold_action"] == ga)).sum())) for ga in _actions]
        rpt.append(f"| Model: {ma} | " + " | ".join(counts) + " |")
    if has_stochasticity:
        rpt.append("\n## Stochasticity (Dual-Run)\n")
        rpt.append("| Metric | Run 1 | Run 2 |")
        rpt.append("|--------|-------|-------|")
        rpt.append(f"| UWAA | {uwaa:.4f} | {uwaa_2:.4f} |")
        rpt.append(f"| Normalized | {normalized:.3f} | {norm_2:.3f} |")
        rpt.append(f"| Items matched | {len(matched_pairs)}/{len(records_1)} |  |")
        rpt.append(f"| Action stability | {stable}/{len(matched_pairs)} ({stable/len(matched_pairs):.0%}) |  |")
        rpt.append(f"| Score range | {min(normalized, norm_2):.2f} \u2013 {max(normalized, norm_2):.2f} |  |")
    rpt.append("\n## Runtime and Cost\n")
    rpt.append("| Metric | Value |")
    rpt.append("|--------|-------|")
    rpt.append(f"| Input tokens | {in_tok:,} |")
    rpt.append(f"| Output tokens | {out_tok:,} |")
    rpt.append(f"| Latency | {latency_ms/1000:.1f}s |")
    rpt.append(f"| Input cost | ${in_cost_nd/1e9:.4f} |")
    rpt.append(f"| Output cost | ${out_cost_nd/1e9:.4f} |")
    rpt.append(f"| Total cost | ${(in_cost_nd+out_cost_nd)/1e9:.4f} |")
    rpt.append("\n## Item Detail\n")
    rpt.append("| Item | Gold Action | Model Decision | Correct | Utility |")
    rpt.append("|------|------------|---------------|---------|---------|")
    for _, row in audit.sort_values("item_id").iterrows():
        cm = "\u2713" if row["is_correct"] else "\u2717"
        rpt.append(f"| {row['item_id']} | {row['gold_action']} | {row['model_decision']} | {cm} | {float(row['utility']):+.1f} |")
    neg = audit[audit["utility"].astype(float) < 0].sort_values("item_id")
    if len(neg) > 0:
        rpt.append(f"\n## Negative Utility Items ({len(neg)})\n")
        for _, row in neg.iterrows():
            rpt.append(f"### {row['item_id']}")
            rpt.append(f"- **Gold action:** {row['gold_action']} | **Model:** {row['model_decision']}")
            rpt.append(f"- **Answer:** {str(row['model_answer'])[:80]}")
            rpt.append(f"- **Utility:** {float(row['utility']):+.1f}\n")
            _j = _justif.get(row['item_id'], '')
            if _j:
                rpt.append(f"- **Justification:** {_j}")
    report_path = os.path.join(OUTPUT_DIR, f"MetaJudge_Abstention_{model_slug}_v6.5.md")
    with open(report_path, "w") as f:
        f.write("\n".join(rpt))
    print(f"Audit report: {report_path}")

In [ ]:
@kbench.task(name="metajudge_abstention_v65", version=6)
def metajudge_abstention_v65(llm) -> float:
    """Selective abstention scored by 5x4 utility matrix. 72 items, 4 actions
    (answer/clarify/verify/abstain). UWAA with answer-rate penalty, normalized."""
    # Prompt: structured output with decision, answer, confidence,
    #   clarification_request, verification_target, abstention_reason
    #   (see metacog_abstention above)
    fb_eval = fb_df[["item_id", "question", "gold_answer", "gold_action"]].copy()

    # Run 1 (scored — cached)
    with kbench.client.enable_cache():
        runs_1 = metacog_abstention.evaluate(
            llm=[llm], evaluation_data=fb_eval,
            n_jobs=8, remove_run_files=True,
            stop_condition=lambda runs: len(runs) == len(fb_eval),
            max_attempts=1,
        )
    records_1 = [r.result for r in runs_1 if isinstance(r.result, dict)]
    if len(records_1) < len(fb_eval):
        print(f"  WARNING: {len(fb_eval)-len(records_1)}/{len(fb_eval)} items failed.")

    # Run 2 (stochasticity — uncached, display only)
    records_2 = []
    try:
        records_2 = [r.result for r in metacog_abstention.evaluate(
            llm=[llm], evaluation_data=fb_eval,
            n_jobs=8, remove_run_files=True,
            stop_condition=lambda runs: len(runs) == len(fb_eval),
            max_attempts=1,
        ) if isinstance(r.result, dict)]
    except Exception as e:
        print(f"  Stochasticity Run 2 failed: {e}")

    if not records_1:
        print("  FATAL: All items failed. Returning score 0.0.")
        return 0.0

    # Score: UWAA with answer-rate penalty, anchor-normalized to [0, 1]
    audit = pd.DataFrame(records_1)
    baseline_rate = compute_baseline_answer_rate(fb_items)
    adjusted_utils, penalty = apply_answer_rate_penalty(
        audit["utility"].tolist(), audit["model_decision"].tolist(), baseline_rate)
    uwaa = compute_uwaa(adjusted_utils)
    normalized = normalize(uwaa, ANCHOR_B_FLOOR, ANCHOR_B_CEIL)

    _report_results(llm, audit, records_1, records_2, len(fb_eval),
                    uwaa, normalized, penalty, baseline_rate,
                    ANCHOR_B_FLOOR, ANCHOR_B_CEIL)
    return normalized

In [ ]:
metajudge_abstention_v65.run(kbench.llm)
%choose metajudge_abstention_v65


## Methodology

Per-item utility from a 5×4 payoff matrix mapping (model action × gold action) → [−1, +1].
UWAA = (mean utility + 1) / 2, normalized to [0, 1]. 72 clean items (12 excluded).
Handles corrective answers on false-presupposition items and acceptable alternative actions.
Dual-run stochasticity check: Run 1 cached (scored), Run 2 independent (display only).
Normalized using anchor floor 0.60 (empirical random baseline) and ceiling 1.0.

For theoretical grounding, statistical methodology, and full citations,
see `docs/benchmark/v5_theoretical_backgrounder.md`.
